# Project 2: SVM with Hinge Loss from Scratch

**Chapter:** 3 — Hinge Loss & Regularization

**Goal:** Replace the perceptron update rule with hinge loss + gradient descent to build a soft-margin SVM. Understand why maximizing the margin matters.

**What you'll build:**
1. Hinge loss function and its gradient
2. Gradient descent training loop with L2 regularization
3. Side-by-side comparison: perceptron boundary vs SVM boundary

**Rules:** No sklearn. NumPy + matplotlib only.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. Data (reused from Project 1)

In [ ]:
def make_data(n=200, margin=1.5, seed=42):
    rng = np.random.default_rng(seed)
    X_pos = rng.normal(loc=[2, 2], scale=0.8, size=(n // 2, 2))
    X_neg = rng.normal(loc=[-2, -2], scale=0.8, size=(n // 2, 2))
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n // 2) + [-1] * (n // 2))
    idx = rng.permutation(n)
    return X[idx], y[idx]

X_train, y_train = make_data(n=200)

def plot_data(X, y, ax=None, title="Data"):
    if ax is None:
        fig, ax = plt.subplots()
    ax.scatter(X[y==1,0], X[y==1,1], c='steelblue', marker='o', label='+1', edgecolors='k', linewidth=0.5, s=50)
    ax.scatter(X[y==-1,0], X[y==-1,1], c='salmon', marker='s', label='-1', edgecolors='k', linewidth=0.5, s=50)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.set_title(title); ax.legend(); ax.set_aspect('equal')
    return ax

def plot_boundary(X, y, w, b, ax=None, title="Boundary", show_margin=False):
    if ax is None:
        fig, ax = plt.subplots()
    plot_data(X, y, ax=ax, title=title)
    xlim = ax.get_xlim()
    if abs(w[1]) > 1e-10:
        x1 = np.linspace(xlim[0]-1, xlim[1]+1, 200)
        ax.plot(x1, -(w[0]*x1 + b)/w[1], 'k-', lw=2, label='boundary')
        if show_margin:
            ax.plot(x1, -(w[0]*x1 + b - 1)/w[1], 'k--', lw=1, alpha=0.5)
            ax.plot(x1, -(w[0]*x1 + b + 1)/w[1], 'k--', lw=1, alpha=0.5)
        ax.set_xlim(xlim)
    ax.legend()
    return ax

plot_data(X_train, y_train, title="Training Data (200 points)")
plt.show()

## 2. Hinge Loss & SVM — Your Turn

### The math

**Hinge loss** for one point $(\mathbf{x}_i, y_i)$:

$$L_i = \max(0,\; 1 - y_i (\mathbf{w} \cdot \mathbf{x}_i + b))$$

**Objective** (average hinge loss + L2 regularization):

$$J(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n} \max(0,\; 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i + b)) + \frac{\lambda}{2} \|\mathbf{w}\|^2$$

**Gradients** (for one point, when $1 - y_i(\mathbf{w}\cdot\mathbf{x}_i + b) > 0$):

$$\frac{\partial L_i}{\partial \mathbf{w}} = -y_i \mathbf{x}_i + \lambda \mathbf{w}, \quad \frac{\partial L_i}{\partial b} = -y_i$$

When the point is correctly classified with margin $\geq 1$: gradient of hinge is zero (only regularization remains for $\mathbf{w}$).

### Your task

Fill in the methods marked `### YOUR CODE ###`.

In [ ]:
class LinearSVM:
    def __init__(self, n_features, lr=0.01, reg_lambda=0.01):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.lr = lr
        self.reg_lambda = reg_lambda
        self.loss_history = []
    
    def hinge_loss(self, X, y):
        """Compute the average hinge loss + L2 regularization over the dataset.
        
        Returns:
            float: J(w, b)
        """
        ### YOUR CODE ###
        raise NotImplementedError("Compute average max(0, 1 - y*(w·x + b)) + λ/2 * ||w||²")
    
    def compute_gradients(self, X, y):
        """Compute gradients of J w.r.t. w and b over the full dataset.
        
        Returns:
            dw: gradient w.r.t. w, shape (n_features,)
            db: gradient w.r.t. b, scalar
        """
        ### YOUR CODE ###
        # For each point: check if it violates the margin (1 - y*(w·x+b) > 0)
        # If yes: gradient contribution is -y*x (for w) and -y (for b)
        # If no: gradient contribution is 0
        # Average over all points, then add regularization term λ*w to dw
        raise NotImplementedError("Compute dw and db")
    
    def train_step(self, X, y):
        """One gradient descent step.
        
        1. Compute gradients
        2. Update w and b: w = w - lr * dw, b = b - lr * db
        3. Record the loss
        """
        ### YOUR CODE ###
        raise NotImplementedError("Gradient descent update")
    
    def predict(self, X):
        """Predict labels +1 or -1."""
        ### YOUR CODE ###
        raise NotImplementedError("sign(X @ w + b)")
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

### Sanity checks

In [ ]:
svm_test = LinearSVM(n_features=2, lr=0.01, reg_lambda=0.0)

# Test 1: hinge loss with zero weights — all points have margin -1, so loss = max(0, 1-0) = 1
loss_zero = svm_test.hinge_loss(X_train, y_train)
assert abs(loss_zero - 1.0) < 0.01, f"With zero weights, avg hinge loss should be ~1.0, got {loss_zero}"
print("✓ Test 1: hinge_loss correct with zero weights")

# Test 2: predict with zero weights should return +1 (sign(0) = +1)
preds = svm_test.predict(X_train[:5])
assert all(p == 1 for p in preds), f"Zero weights should predict all +1"
print("✓ Test 2: predict correct with zero weights")

# Test 3: one training step should reduce the loss
svm_test2 = LinearSVM(n_features=2, lr=0.1, reg_lambda=0.0)
loss_before = svm_test2.hinge_loss(X_train, y_train)
svm_test2.train_step(X_train, y_train)
loss_after = svm_test2.hinge_loss(X_train, y_train)
assert loss_after < loss_before, f"Loss should decrease after one step: {loss_before:.4f} -> {loss_after:.4f}"
print(f"✓ Test 3: loss decreased {loss_before:.4f} → {loss_after:.4f}")

print("\n🎉 All checks passed!")

## 3. Train & Visualize

In [ ]:
svm = LinearSVM(n_features=2, lr=0.01, reg_lambda=0.01)

for epoch in range(1, 501):
    svm.train_step(X_train, y_train)
    if epoch <= 5 or epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  |  loss: {svm.loss_history[-1]:.4f}  |  acc: {svm.accuracy(X_train, y_train):.1%}")

print(f"\nFinal: w = {svm.w.round(4)}, b = {svm.b:.4f}")

In [ ]:
plt.plot(svm.loss_history)
plt.xlabel('Epoch'); plt.ylabel('Hinge Loss + Regularization')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plot_boundary(X_train, y_train, svm.w, svm.b, title="SVM Boundary with Margin", show_margin=True)
plt.tight_layout()
plt.show()

## 4. Experiments

### A: Regularization strength
Try `reg_lambda` = 0.001, 0.01, 0.1, 1.0. How does the boundary change?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for lam, ax in zip([0.001, 0.01, 0.1, 1.0], axes):
    m = LinearSVM(n_features=2, lr=0.01, reg_lambda=lam)
    for _ in range(500):
        m.train_step(X_train, y_train)
    plot_boundary(X_train, y_train, m.w, m.b, ax=ax,
                  title=f"λ={lam}, acc={m.accuracy(X_train, y_train):.0%}", show_margin=True)
plt.tight_layout()
plt.show()

**Your observation:** *(What happens to the margin as λ increases?)*

...

### B: Perceptron vs SVM — side by side

Copy your Perceptron class from Project 1 and compare the two boundaries.

In [ ]:
# Paste your Perceptron class from Project 1 here, train it,
# then plot both boundaries side by side:
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
# plot_boundary(X_train, y_train, perceptron.w, perceptron.b, ax=ax1, title="Perceptron")
# plot_boundary(X_train, y_train, svm.w, svm.b, ax=ax2, title="SVM", show_margin=True)


**Your observation:** *(Which boundary has more margin? Which is more stable across shuffles?)*

...

## 5. What You Learned

1. What does the hinge loss penalize that the perceptron doesn't?
2. What is the role of λ (regularization)?
3. Why does the SVM boundary tend to be more centered between the classes?

---
**Next project:** Linear Regression — same gradient descent, different loss function.